# Long-read Assembly Statistics


In [ ]:
# import necessary modules
import pandas as pd
import seaborn as sns; sns.set()
import matplotlib.pyplot as plt
import numpy as np

sns.set_style("ticks",{'axes.grid' : True})
sns.set_palette("colorblind")

plt.rcParams["axes.linewidth"] = 1.5
plt.rcParams["xtick.major.width"] = 1.5
plt.rcParams["ytick.major.width"] = 1.5
plt.rcParams["xtick.major.size"] = 8
plt.rcParams["ytick.major.size"] = 8
plt.rcParams["axes.titlepad"] = 20

plt.rcParams['svg.fonttype'] = 'none'
plt.rcParams["axes.titlesize"] = 30
plt.rcParams['axes.labelsize'] = 23.5
plt.rcParams['xtick.labelsize'] = 18
plt.rcParams['ytick.labelsize'] = 18
plt.rcParams['legend.fontsize'] = 18
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Liberation Sans']
plt.rcParams['text.usetex'] = False
plt.rcParams["savefig.dpi"]=300


In [ ]:
input_statistics_quast=snakemake.params.input_quast_report
SAMPLING=snakemake.params.sampling

output_summary_html=snakemake.output.summary_html
output_contig_number_png=snakemake.output.contig_number_png
output_contig_number_svg=snakemake.output.contig_number_svg
output_total_length_png=snakemake.output.total_length_png
output_total_length_svg=snakemake.output.total_length_svg
output_n50_png=snakemake.output.n50_png
output_n50_svg=snakemake.output.n50_svg


## Parse QUAST transposed report


In [ ]:
df=pd.read_csv(input_statistics_quast, sep="\t")
df=df.copy()

if "Assembly" not in df.columns:
    df=df.rename(columns={df.columns[0]: "Assembly"})

df["Assembly"]=df["Assembly"].astype(str)
df["Assembly"]=df["Assembly"].str.replace("_corrected_scaffolds_pilon." + SAMPLING, "", regex=False)
df["Assembly"]=df["Assembly"].str.replace("_contigs_hifiasm." + SAMPLING, "", regex=False)
df["Assembly"]=df["Assembly"].str.replace("_contigs_flye." + SAMPLING, "", regex=False)
df["Assembly"]=df["Assembly"].str.replace("racon_", "", regex=False)
df["Assembly"]=df["Assembly"].str.replace("_contigs_2_flye." + SAMPLING, "", regex=False)
df["Assembly"]=df["Assembly"].str.replace("polypolish_", "", regex=False)
df=df.sort_values(by="Assembly")

numeric_cols=[col for col in df.columns if col != "Assembly"]
for col in numeric_cols:
    df[col]=pd.to_numeric(df[col], errors="ignore")

df


In [ ]:
summary_cols=[col for col in ["Assembly", "# contigs (>= 1000 bp)", "Total length", "Total length (>= 1000 bp)", "N50", "Largest contig", "GC (%)"] if col in df.columns]
summary_df=df[summary_cols].copy()
summary_df_out=summary_df.style.background_gradient(cmap="RdYlGn").to_html()
with open(output_summary_html, "w") as fp:
    fp.write(summary_df_out)
summary_df


## Number of contigs


In [ ]:
contig_cols=[col for col in ["# contigs (>= 1000 bp)", "# contigs (>= 10000 bp)", "# contigs (>= 25000 bp)", "# contigs (>= 50000 bp)"] if col in df.columns]
fig_width=max(8, 0.5*len(df))
fig, ax = plt.subplots(figsize=(fig_width, 10))

plot_df=df[["Assembly"] + contig_cols].copy()
plot_df=plot_df.melt(id_vars="Assembly", var_name="Metric", value_name="Count")
sns.barplot(data=plot_df, x="Assembly", y="Count", hue="Metric", ax=ax)

ax.set_xlabel("Assembly")
ax.set_ylabel("# contigs")
ax.set_yscale("log")
ax.set_xticklabels(ax.get_xticklabels(), rotation=90)
ax.legend(loc='center left', bbox_to_anchor=(1, 0.5))

fig.savefig(output_contig_number_png, format="png", bbox_inches="tight", transparent=True)
fig.savefig(output_contig_number_svg, format="svg", bbox_inches="tight", transparent=True)
plt.show()


## Total assembly length


In [ ]:
length_cols=[col for col in ["Total length", "Total length (>= 1000 bp)", "Total length (>= 10000 bp)", "Total length (>= 25000 bp)", "Total length (>= 50000 bp)"] if col in df.columns]
fig_width=max(8, 0.5*len(df))
fig, ax = plt.subplots(figsize=(fig_width, 10))

plot_df=df[["Assembly"] + length_cols].copy()
plot_df=plot_df.melt(id_vars="Assembly", var_name="Metric", value_name="Length")
plot_df["Length_Mbp"]=plot_df["Length"]/1000000
sns.barplot(data=plot_df, x="Assembly", y="Length_Mbp", hue="Metric", ax=ax)

ax.set_xlabel("Assembly")
ax.set_ylabel("Total length (Mbp)")
ax.set_xticklabels(ax.get_xticklabels(), rotation=90)
ax.legend(loc='center left', bbox_to_anchor=(1, 0.5))

fig.savefig(output_total_length_png, format="png", bbox_inches="tight", transparent=True)
fig.savefig(output_total_length_svg, format="svg", bbox_inches="tight", transparent=True)
plt.show()


## N50


In [ ]:
fig_width=max(8, 0.5*len(df))
fig, ax = plt.subplots(figsize=(fig_width, 10))

plot_df=df[["Assembly", "N50"]].copy()
plot_df["N50_kbp"]=plot_df["N50"]/1000
sns.barplot(data=plot_df, x="Assembly", y="N50_kbp", ax=ax)

ax.set_xlabel("Assembly")
ax.set_ylabel("N50 (kbp)")
ax.set_xticklabels(ax.get_xticklabels(), rotation=90)

fig.savefig(output_n50_png, format="png", bbox_inches="tight", transparent=True)
fig.savefig(output_n50_svg, format="svg", bbox_inches="tight", transparent=True)
plt.show()
